In [1]:
!pip install pymupdf langchain langchain-experimental langchain-huggingface faiss-cpu groq sentence-transformers rank_bm25


[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import fitz
import re
import pathlib

def is_table_of_contents(page_text):
    text = page_text.strip()
    if not text:
        return False

    lines = text.split("\n")
    header_match = any(re.search(r"^(table of contents|contents)\b", line.lower()) for line in lines[:3])

    toc_line_count = 0
    for line in lines:
        if "...." in line or re.search(r"\.{3,}\s*\d+$", line):
            toc_line_count += 1

    if header_match and toc_line_count >= 1:
        return True

    if len(lines) > 5:
        lines_ending_with_num = sum(1 for line in lines if re.search(r"\s+\d+$", line.strip()))
        if (lines_ending_with_num / len(lines)) > 0.4 and toc_line_count >= 2:
            return True

    return False

def extract_and_filter_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    cleaned_text_pages = []
    skipped_pages = []

    for page_num in range(len(doc)):
        page = doc[page_num]
        page_text = page.get_text()

        if is_table_of_contents(page_text):
            skipped_pages.append(page_num + 1)
            continue

        cleaned_text_pages.append(page_text)

    print(f"[{pdf_path.name}] Skipped TOC pages: {skipped_pages}")
    return "\n".join(cleaned_text_pages)

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

DATA_DIR = r"C:\Users\aksha\Desktop\rag_stock_market\data\raw"
raw_pdf_texts = {}
raw_pdf_pages = {}

pdf_files = list(pathlib.Path(DATA_DIR).glob("*.pdf"))
for pdf_path in pdf_files:
    raw_pdf_texts[pdf_path.name] = extract_and_filter_pdf(pdf_path)
    source_doc = fitz.open(pdf_path)
    raw_pdf_pages[pdf_path.name] = [
        {"page": page_num + 1, "text": source_doc[page_num].get_text()}
        for page_num in range(len(source_doc))
    ]
    source_doc.close()

print("Creating smaller overlapping chunks with page metadata...")
chunker = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=300,
    separators=["\n\n", "\n", ". ", " ", ""]
)

semantic_chunks = []
semantic_metadata = []

print("Running overlapping chunking across documents...")
for filename, pages in raw_pdf_pages.items():
    for page_info in pages:
        page_text = page_info["text"]
        if not page_text.strip():
            continue
        docs = chunker.create_documents([page_text])
        for doc in docs:
            semantic_chunks.append(doc.page_content)
            semantic_metadata.append({
                "chunk_id": len(semantic_chunks) - 1,
                "filename": filename,
                "page": page_info["page"]
            })

print(f"Generated {len(semantic_chunks)} smaller overlapping chunks.")


[amfi_mutual_funds_made_simple_2026.pdf] Skipped TOC pages: []
[bse_ipo_investor_guide.pdf] Skipped TOC pages: []
[bse_margin_reporting_guidelines.pdf] Skipped TOC pages: []
[bse_peak_margin_faq.pdf] Skipped TOC pages: []
[bse_rights_issue_investor_guide.pdf] Skipped TOC pages: []
[bse_t0_settlement_faq.pdf] Skipped TOC pages: []
[cdsl_iepf_investor_guide_2025.pdf] Skipped TOC pages: []
[cdsl_investor_booklet_2025.pdf] Skipped TOC pages: []
[common_booklet_securities_market_2021.pdf] Skipped TOC pages: []
[introduction_of_t1_rolling_settlement_cycle_in_equity_market_nov242022_2022.pdf] Skipped TOC pages: []
[nsdl_eguide_demat_account_holders_2026.pdf] Skipped TOC pages: [2, 3, 4]
[nse_basic_faqs_secondary_market_2023.pdf] Skipped TOC pages: [2]
[nse_faq_on_asba_ipo_for_investors_2023.pdf] Skipped TOC pages: []
[nse_parichay_financial_literacy_2026.pdf] Skipped TOC pages: []
[nse_pragati_basics_capital_markets_2024.pdf] Skipped TOC pages: []
[nse_pratham_financial_planning_2026.pdf] Ski

In [4]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

print("Encoding semantic chunks...")
encoder = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = encoder.encode(semantic_chunks, show_progress_bar=True).astype('float32')

dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

# Global variables for dense retrieval
chunks = semantic_chunks
metadata = semantic_metadata

# Initialize BM25 keyword index
print("Initializing BM25 index...")
tokenized_corpus = [chunk.lower().split() for chunk in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print(f"Successfully indexed {index.ntotal} chunks into FAISS and BM25!")

c:\Users\aksha\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Encoding semantic chunks...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8562.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 217/217 [02:38<00:00,  1.37it/s]


Initializing BM25 index...
Successfully indexed 6917 chunks into FAISS and BM25!


In [5]:
from sentence_transformers import CrossEncoder

print("Loading Cross-Encoder reranker model...")
reranker_model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded successfully!")

Loading Cross-Encoder reranker model...


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3280.88it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded successfully!


In [6]:
from groq import Groq
import os
from getpass import getpass

api_key = os.getenv("GROQ_API_KEY") or getpass("Enter your Groq API key: ")
if not api_key:
    raise RuntimeError("GROQ_API_KEY is required to run the Groq evaluation.")
client = Groq(api_key=api_key)


def build_prompt(query, retrieved_chunks):
    context = "\n\n---\n\n".join([c["text"] for c in retrieved_chunks])

    prompt = f"""You are a precise assistant for the Indian stock market.
Answer the question based ONLY on the provided context documents.
If the answer is not contained in the context, say "I cannot answer this based on the provided documents." Do not guess.

Context Documents:
{context}

Question: {query}
Answer:"""
    return prompt

def generate_answer_with_groq(query, retrieved_chunks):
    prompt = build_prompt(query, retrieved_chunks)
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"

print("Groq-powered pipeline functions ready!")

Groq-powered pipeline functions ready!


In [7]:
from groq import Groq
import os
from getpass import getpass
import base64
import numpy as np

api_key = os.getenv("GROQ_API_KEY") or getpass("Enter your Groq API key: ")
if not api_key:
    raise RuntimeError("GROQ_API_KEY is required to run the Groq evaluation.")
client = Groq(api_key=api_key)

TEXT_MODEL = "openai/gpt-oss-20b"
VISION_MODEL = os.getenv("GROQ_VISION_MODEL", "qwen/qwen3.8-27b")
IMAGE_PAGE_LIMIT = 1
IMAGE_RENDER_SCALE = 1.0

def expanded_hybrid_retrieve_with_rerank(query, k=5, initial_pool=80, candidate_k=50, bm25_weight=0.5, vector_weight=0.5):
    """Retrieve and rerank chunks without corpus-specific routing or query tuning."""
    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:candidate_k]

    query_vector = encoder.encode([query]).astype("float32")
    _, vector_indices = index.search(query_vector, min(candidate_k, index.ntotal))
    vector_top_indices = vector_indices[0]

    # Equal-weight reciprocal-rank fusion keeps retrieval behavior independent
    # of benchmark topics and avoids hand-tuned source or term boosts.
    rrf_k = 60
    combined_rrf_scores = {}
    for rank, idx in enumerate(bm25_top_indices):
        idx = int(idx)
        combined_rrf_scores[idx] = combined_rrf_scores.get(idx, 0.0) + bm25_weight / (rrf_k + rank + 1)
    for rank, idx in enumerate(vector_top_indices):
        idx = int(idx)
        combined_rrf_scores[idx] = combined_rrf_scores.get(idx, 0.0) + vector_weight / (rrf_k + rank + 1)

    candidate_indices = sorted(
        combined_rrf_scores,
        key=lambda idx: (-combined_rrf_scores[idx], idx),
    )[:initial_pool]
    candidate_results = [{
        "chunk_id": idx,
        "text": chunks[idx],
        "source": metadata[idx]["filename"],
        "page": metadata[idx].get("page"),
    } for idx in candidate_indices]
    if not candidate_results:
        return []

    cross_input = [[query, doc["text"]] for doc in candidate_results]
    rerank_scores = reranker_model.predict(
        cross_input,
        batch_size=16,
        show_progress_bar=False,
    )
    for i, doc in enumerate(candidate_results):
        doc["score"] = float(rerank_scores[i])

    return sorted(
        candidate_results,
        key=lambda doc: (-doc["score"], doc["chunk_id"]),
    )[:k]

def build_prompt(query, retrieved_chunks):
    context_parts = []
    for chunk in retrieved_chunks:
        page_label = f", page {chunk['page']}" if chunk.get("page") else ""
        context_parts.append(f"Source: {chunk['source']}{page_label}\n{chunk['text']}")
    context = "\n\n---\n\n".join(context_parts)
    return f"""You are a precise assistant for the Indian stock market.
Answer the question based ONLY on the provided context documents and attached page images.
Use images only to clarify diagrams, candlestick shapes, labels, or other visual information. Do not infer trading certainty or guaranteed profitability from an image.
If the answer is not contained in the context or images, say "I cannot answer this based on the provided documents." Do not guess.

Context Documents:
{context}

Question: {query}
Answer:"""

def _get_retrieved_page_images(retrieved_chunks):
    image_parts = []
    seen_pages = set()
    for chunk in retrieved_chunks:
        source = chunk.get("source")
        page_number = chunk.get("page")
        if not source or not page_number or (source, page_number) in seen_pages:
            continue
        pdf_path = pathlib.Path(DATA_DIR) / source
        if not pdf_path.exists():
            continue
        doc = fitz.open(pdf_path)
        try:
            page = doc.load_page(int(page_number) - 1)
            if not page.get_images(full=True):
                continue
            pixmap = page.get_pixmap(matrix=fitz.Matrix(IMAGE_RENDER_SCALE, IMAGE_RENDER_SCALE), alpha=False)
            encoded = base64.b64encode(pixmap.tobytes("png")).decode("ascii")
            image_parts.append({"source": source, "page": page_number, "data": encoded})
            seen_pages.add((source, page_number))
            if len(image_parts) >= IMAGE_PAGE_LIMIT:
                break
        finally:
            doc.close()
    return image_parts

def generate_answer_with_groq(query, retrieved_chunks):
    prompt = build_prompt(query, retrieved_chunks)
    images = _get_retrieved_page_images(retrieved_chunks)
    content = [{"type": "text", "text": prompt}]
    for image in images:
        content.append({
            "type": "text",
            "text": f"Attached image source: {image['source']}, page {image['page']}"
        })
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{image['data']}"}
        })
    model = VISION_MODEL if images else TEXT_MODEL
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": content}],
            temperature=0.0
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error using {model}: {e}"

print(f"Improved Groq text+PDF-image pipeline ready. Vision model: {VISION_MODEL}")


Improved Groq text+PDF-image pipeline ready. Vision model: qwen/qwen3.8-27b


In [8]:
test_query = "What is the maximum brokerage a stockbroker can charge?"
reranked_results = expanded_hybrid_retrieve_with_rerank(test_query, k=3)

print(f"Question: {test_query}\n")
print("Top Retrieved Chunks:")
for i, c in enumerate(reranked_results):
    print(f"[{i+1}] Chunk {c['chunk_id']} | Source: {c['source']} | Page: {c.get('page')} (Score: {c['score']:.4f})")
    print(f"Snippet: {c['text'][:150]}...\n")

answer = generate_answer_with_groq(test_query, reranked_results)
print(f"Generated Answer:\n{answer}")

Question: What is the maximum brokerage a stockbroker can charge?

Top Retrieved Chunks:
[1] Chunk 344 | Source: nse_basic_faqs_secondary_market_2023.pdf | Page: 15 (Score: 8.9458)
Snippet: 24. What is the maximum brokerage that a broker can charge? 
 
Brokerage can be charged as may be mutually agreed between member & client subject to 
...

[2] Chunk 3519 | Source: sebi_kyc_account_opening.pdf | Page: 25 (Score: 8.3429)
Snippet: Brokerage Charges
25
https://investor.sebi.gov.in
As per the BSE & NSE Bye Laws, the brokerage that can be charged by a stock
broker are as under:
Ca...

[3] Chunk 6035 | Source: sebi_stock_brokers_master_circular_2025.pdf | Page: 49 (Score: -0.1305)
Snippet: 49 
 
broker, all stockbrokers (subject to maximum of 
5) shall be considered as QSBs, if their individual 
compliance score is equal to or more than ...

Generated Answer:
Based on the provided documents and images, the maximum brokerage a stockbroker can charge is as follows:

*   **Capital Market Seg

In [9]:
import json
import numpy as np
from tqdm.auto import tqdm

EVAL_FILE = r"C:\Users\aksha\Desktop\rag_stock_market\Golden_set_300.json"
# The benchmark contains fixed relevant_chunk_ids for exact chunk-level evaluation.

with open(EVAL_FILE, "r", encoding="utf-8") as handle:
    golden_set = json.load(handle)

def evaluate_configuration(k=5, split="test", **retrieval_kwargs):
    """Evaluate a fixed configuration against one benchmark split."""
    evaluated_items = [
        item for item in golden_set
        if item.get("answerable") and item.get("evaluation_split") == split
    ]
    if not evaluated_items:
        raise ValueError(f"No answerable items in evaluation split: {split}")

    hits = 0
    reciprocal_ranks = []
    misses = []
    for item in tqdm(evaluated_items, desc=f"Evaluating {split}", unit="question"):
        relevant_chunk_ids = set(item.get("relevant_chunk_ids", []))
        if not relevant_chunk_ids:
            raise ValueError(f"Missing relevant_chunk_ids for {item['id']}")
        retrieved = expanded_hybrid_retrieve_with_rerank(
            item["question"],
            k=k,
            **retrieval_kwargs,
        )
        rank = next(
            (index + 1 for index, chunk in enumerate(retrieved)
             if chunk.get("chunk_id") in relevant_chunk_ids),
            None,
        )
        if rank is None:
            reciprocal_ranks.append(0.0)
            misses.append((item["id"], sorted(relevant_chunk_ids), [
                chunk.get("chunk_id") for chunk in retrieved
            ]))
        else:
            hits += 1
            reciprocal_ranks.append(1.0 / rank)

    return hits / len(evaluated_items), float(np.mean(reciprocal_ranks)), misses

# Keep tuning local and bounded: 3 candidate depths x 5 fusion weights = 15 runs.
# The reranking pool remains at its established default to avoid unnecessary work.
search_space = [
    {"candidate_k": candidate_k, "initial_pool": 80, "bm25_weight": bm25_weight, "vector_weight": 1.0 - bm25_weight}
    for candidate_k in (30, 50, 80)
    for bm25_weight in (0.0, 0.25, 0.5, 0.75, 1.0)
]

development_results = []
for configuration in search_space:
    metrics = evaluate_configuration(split="development", **configuration)
    development_results.append({
        **configuration,
        "hit_at_5": metrics[0],
        "mrr_at_5": metrics[1],
    })

best_configuration = max(
    development_results,
    key=lambda result: (result["mrr_at_5"], result["hit_at_5"], -result["candidate_k"], -result["initial_pool"]),
)
print(f"Best development configuration: {best_configuration}")

# Evaluate the selected development configuration once on untouched test data.
held_out_results = evaluate_configuration(
    split="test",
    candidate_k=best_configuration["candidate_k"],
    initial_pool=best_configuration["initial_pool"],
    bm25_weight=best_configuration["bm25_weight"],
    vector_weight=best_configuration["vector_weight"],
)
print(f"HELD-OUT CHUNK Hit@5: {held_out_results[0] * 100:.2f}%")
print(f"HELD-OUT CHUNK MRR@5: {held_out_results[1]:.4f}")


Evaluating development:   0%|          | 0/158 [00:00<?, ?question/s]

Evaluating development:  20%|█▉        | 31/158 [01:24<05:44,  2.71s/question]


KeyboardInterrupt: 